# Nuclear Data — Exploratory Analysis

This notebook performs an initial exploration of two open datasets on nuclear energy production (Our World in Data / Energy Institute).

**Goals:**
- Understand the structure and quality of the raw data
- Identify the countries that have historically produced nuclear energy
- Rank the top producers by average historical output (TWh)

**Datasets used:**
- `nuclear-energy-generation.csv` — annual nuclear electricity generation by country (1965–2025)
- `owid-energy-data.csv` — full energy mix dataset with 130 variables across all countries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

# Logging configuration — force=True ensures clean setup in notebook kernels
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s — %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

logger.info("Libraries loaded successfully.")

INFO — Libraries loaded successfully.


## 1. Loading the datasets

We load both raw CSV files into separate DataFrames. `df_gen` is the primary source for nuclear production figures; `df_owid` provides the broader energy context (consumption mix, per-capita figures, GDP) and will be used for deeper analysis in the next notebook.

In [2]:
df_gen  = pd.read_csv('../data/raw/nuclear-energy-generation.csv')
df_owid = pd.read_csv('../data/raw/owid-energy-data.csv')

logger.info("Dataset 1 — nuclear-energy-generation: %d rows, %d columns", len(df_gen), len(df_gen.columns))
logger.info("Dataset 2 — owid-energy-data: %d rows, %d columns", len(df_owid), len(df_owid.columns))
logger.info("Columns df_gen: %s", list(df_gen.columns))

INFO — Dataset 1 — nuclear-energy-generation: 10064 rows, 4 columns
INFO — Dataset 2 — owid-energy-data: 23232 rows, 130 columns
INFO — Columns df_gen: ['Entity', 'Code', 'Year', 'Nuclear']


## 2. Data structure and quality

We inspect both datasets for shape, column names, value ranges, and missing data.

`df_gen` is compact (4 columns, ~10k rows) and nearly complete — only the `Code` column has missing values, corresponding to regional aggregates rather than individual countries. `df_owid` is much wider (130 columns) with significant sparsity in nuclear-specific columns, which is expected since most countries have never produced nuclear electricity.

In [3]:
logger.info("First 5 rows:")
display(df_gen.head())

logger.info("Last 5 rows:")
display(df_gen.tail())

logger.info("Year range: %d — %d", df_gen['Year'].min(), df_gen['Year'].max())
logger.info("Unique countries/entities: %d", df_gen['Entity'].nunique())

logger.info("Missing values per column:")
display(df_gen.isnull().sum())

INFO — First 5 rows:


,Entity,Code,Year,Nuclear
0,ASEAN (Ember),NaN,2000,0.0
1,ASEAN (Ember),NaN,2001,0.0
2,ASEAN (Ember),NaN,2002,0.0
3,ASEAN (Ember),NaN,2003,0.0
4,ASEAN (Ember),NaN,2004,0.0


INFO — Last 5 rows:


,Entity,Code,Year,Nuclear
10059,Zimbabwe,ZWE,2020,0.0
10060,Zimbabwe,ZWE,2021,0.0
10061,Zimbabwe,ZWE,2022,0.0
10062,Zimbabwe,ZWE,2023,0.0
10063,Zimbabwe,ZWE,2024,0.0


INFO — Year range: 1965 — 2025
INFO — Unique countries/entities: 249
INFO — Missing values per column:


Entity        0
Code       1091
Year          0
Nuclear       0
dtype: int64

In [4]:
# Select only relevant nuclear columns
nuclear_cols = ['country', 'year', 'population', 'gdp',
                'nuclear_electricity', 'nuclear_share_elec',
                'nuclear_share_energy', 'nuclear_elec_per_capita']

df_nuclear = df_owid[nuclear_cols].copy()

logger.info("Nuclear dataset: %d rows, %d columns", len(df_nuclear), len(df_nuclear.columns))

logger.info("First 5 rows:")
display(df_nuclear.head())

logger.info("Missing values:")
display(df_nuclear.isnull().sum())

logger.info("Summary statistics:")
display(df_nuclear.describe())

INFO — Nuclear dataset: 23232 rows, 8 columns
INFO — First 5 rows:


,country,year,population,gdp,nuclear_electricity,nuclear_share_elec,nuclear_share_energy,nuclear_elec_per_capita
0,ASEAN (Ember),2000,NaN,NaN,0.0,0.0,NaN,NaN
1,ASEAN (Ember),2001,NaN,NaN,0.0,0.0,NaN,NaN
2,ASEAN (Ember),2002,NaN,NaN,0.0,0.0,NaN,NaN
3,ASEAN (Ember),2003,NaN,NaN,0.0,0.0,NaN,NaN
4,ASEAN (Ember),2004,NaN,NaN,0.0,0.0,NaN,NaN


INFO — Missing values:


country                        0
year                           0
population                  4472
gdp                        11452
nuclear_electricity        13169
nuclear_share_elec         15677
nuclear_share_energy       16853
nuclear_elec_per_capita    14260
dtype: int64

INFO — Summary statistics:


,year,population,gdp,nuclear_electricity,nuclear_share_elec,nuclear_share_energy,nuclear_elec_per_capita
count,23232.000000,1.876000e+04,1.178000e+04,10063.000000,7555.000000,6379.000000,8972.000000
mean,1975.937543,1.057591e+08,4.257565e+11,90.840326,5.814833,2.811101,313.581786
std,35.234386,4.697550e+08,3.507870e+12,342.292969,13.080006,5.979653,961.634752
min,1900.000000,1.776000e+03,1.642060e+08,0.000000,0.000000,0.000000,0.000000
25%,1949.000000,1.600333e+06,1.426394e+10,0.000000,0.000000,0.000000,0.000000
50%,1985.000000,6.950666e+06,4.357680e+10,0.000000,0.000000,0.000000,0.000000
75%,2005.000000,2.575218e+07,1.830576e+11,3.977000,2.248000,2.137000,0.000000
max,2025.000000,8.161973e+09,1.301126e+14,2778.760000,88.011000,41.661000,8908.020000


## 3. Filtering for real countries

The dataset includes regional and economic aggregates (e.g. `ASEAN (Ember)`, `World`) that do not have a valid ISO alpha-3 country code. Including them would distort any country-level ranking or average.

We filter `df_gen` to keep only rows where `Code` matches the pattern `^[A-Z]{3}$` (exactly three uppercase letters), which is the standard ISO 3166-1 alpha-3 format for sovereign states and territories.

In [5]:
import re

# Filter real countries only (ISO alpha-3 code — exactly 3 uppercase letters)
mask = df_gen['Code'].str.match(r'^[A-Z]{3}$', na=False)
df_countries = df_gen[mask].copy()

top10 = (df_countries.groupby('Entity')['Nuclear']
         .mean()
         .sort_values(ascending=False)
         .head(10)
         .round(1))

logger.info("Top 10 countries by average historical nuclear output (TWh):")
display(top10)

logger.info("Total countries with ISO code: %d", df_countries['Entity'].nunique())
logger.info("Year range after filter: %d — %d", df_countries['Year'].min(), df_countries['Year'].max())

INFO — Top 10 countries by average historical nuclear output (TWh):


Entity
United States     533.3
France            270.3
Russia            154.8
Japan             136.5
Germany            92.6
Ukraine            78.0
South Korea        76.8
China              73.5
Canada             65.0
United Kingdom     58.1
Name: Nuclear, dtype: float64

INFO — Total countries with ISO code: 210
INFO — Year range after filter: 1965 — 2025


In [6]:
# All unique entities passing the ISO filter
iso_codes = df_countries[['Entity', 'Code']].drop_duplicates().sort_values('Code')

logger.info("Total entities with 3-letter ISO code: %d", len(iso_codes))
logger.info("Full list:")
display(iso_codes.to_string())


INFO — Total entities with 3-letter ISO code: 210
INFO — Full list:


"                                 Entity Code\n469                               Aruba  ABW\n25                          Afghanistan  AFG\n300                              Angola  AGO\n192                             Albania  ALB\n9347               United Arab Emirates  ARE\n384                           Argentina  ARG\n444                             Armenia  ARM\n276                      American Samoa  ASM\n360                 Antigua and Barbuda  ATG\n636                           Australia  AUS\n696                             Austria  AUT\n757                          Azerbaijan  AZE\n1481                            Burundi  BDI\n1004                            Belgium  BEL\n1089                              Benin  BEN\n1457                       Burkina Faso  BFA\n880                          Bangladesh  BGD\n1396                           Bulgaria  BGR\n820                             Bahrain  BHR\n797                             Bahamas  BHS\n1216             Bosnia and Herze

> **Note:** the full list above includes all 210 entities with a valid ISO code, the vast majority of which have zero nuclear production across all years. The next cell narrows the focus to the **top 10 producers** ranked by their historical average output in TWh.

In [7]:
# Top 10 real countries by average historical nuclear output
top10 = (df_countries.groupby('Entity')['Nuclear']
         .mean()
         .sort_values(ascending=False)
         .head(10)
         .round(1))

logger.info("Top 10 real countries by average historical nuclear output (TWh):")
display(top10)

INFO — Top 10 real countries by average historical nuclear output (TWh):


Entity
United States     533.3
France            270.3
Russia            154.8
Japan             136.5
Germany            92.6
Ukraine            78.0
South Korea        76.8
China              73.5
Canada             65.0
United Kingdom     58.1
Name: Nuclear, dtype: float64

## 4. Countries that have ever produced nuclear energy

Rather than looking at averages, we now ask a binary question: which countries have recorded **at least one year** with nuclear production greater than zero? This gives us the true count of nations that have operated a nuclear power plant at some point in history.

In [8]:
# Countries that have EVER produced nuclear energy
nuclear_countries = (df_countries[df_countries['Nuclear'] > 0]
                     ['Entity']
                     .unique())

logger.info("Countries that have EVER produced nuclear energy: %d", len(nuclear_countries))
logger.info("Full list:")
for country in sorted(nuclear_countries):
    logger.info("  %s", country)

INFO — Countries that have EVER produced nuclear energy: 36
INFO — Full list:
INFO —   Argentina
INFO —   Armenia
INFO —   Belarus
INFO —   Belgium
INFO —   Brazil
INFO —   Bulgaria
INFO —   Canada
INFO —   China
INFO —   Czechia
INFO —   Finland
INFO —   France
INFO —   Germany
INFO —   Hungary
INFO —   India
INFO —   Iran
INFO —   Italy
INFO —   Japan
INFO —   Kazakhstan
INFO —   Lithuania
INFO —   Mexico
INFO —   Netherlands
INFO —   Pakistan
INFO —   Romania
INFO —   Russia
INFO —   Slovakia
INFO —   Slovenia
INFO —   South Africa
INFO —   South Korea
INFO —   Spain
INFO —   Sweden
INFO —   Switzerland
INFO —   Taiwan
INFO —   Ukraine
INFO —   United Arab Emirates
INFO —   United Kingdom
INFO —   United States


## 5. Exploratory conclusions

Key findings from this initial exploration:

- **36 countries** have ever produced nuclear electricity — out of 210 with a valid ISO code, that is less than 18% of all nations
- **The USA dominates** with a historical average of 533 TWh, more than double France (270 TWh), which ranks second
- **The dataset spans 1965–2025**, covering the full arc of civil nuclear history from early commercial plants to the present day
- **Data quality is good** for `df_gen` (zero missing values in the core columns); `df_owid` has significant sparsity in nuclear-specific columns (`nuclear_share_elec` is missing for ~68% of rows), consistent with the small number of nuclear-producing nations

These findings set the baseline for the visualisations and counterfactual analysis developed in notebook `02_visualizations.ipynb`.